# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa - Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² Mental Health Survey dataset (collected from Kilifi County, Kenya) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant metadata schema.

The dataset contains survey results covering demographics and psychological assessments such as GAD-7, PHQ-9, and PSQ scores. We'll load the data, review available record sets and fields by their `@id`, and perform basic exploratory analysis.

### Dataset Source
The dataset Croissant schema is hosted at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We'll load the Croissant schema from the given URL and initialize the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access Croissant metadata
metadata = dataset.metadata

print(f"Dataset name: {metadata.name if hasattr(metadata, 'name') else metadata['name']}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else metadata['description']}")

## 2. Data Overview

Let's discover the available record sets (tables) as well as their fields and columns. We will reference all entities by their Croissant `@id` fields.

If your dataset contains multiple record sets, list their `@id`s. For each record set, enumerate available fields, their descriptions, and their associated `@id`s.

In [ ]:
# List record sets, fields, and their @ids
record_sets = dataset.record_sets

print("Available record sets and fields:")
overview = {}
for rs in record_sets:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', None))
    rs_name = getattr(rs, 'name', 'N/A')
    print(f"  RecordSet name: {rs_name}")
    print(f"    @id: {rs_id}")
    
    # List fields in the record set
    print(f"    Fields:")
    overview[rs_id] = []
    for field in getattr(rs, 'fields', []):
        field_id = getattr(field, '@id', getattr(field, 'id', None))
        field_name = getattr(field, 'name', 'N/A')
        overview[rs_id].append(field_id)
        print(f"      - {field_name} (id: {field_id})")
    print()

# Show the overview structure for further reference
pprint.pprint(overview)

## 3. Data Extraction

Now, load data from one or more record sets into pandas DataFrames for analysis. We'll use the `@id` of each record set as keys. 

For the FAIR² Mental Health Survey dataset, you'll likely see a table such as `cr:RecordSet/mental_health_survey` (actual `@id` will be displayed above).


In [ ]:
# Specify the record sets to load (use @ids from overview above)
record_set_ids = list(overview.keys())
dataframes = dict()

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"{rs_id} columns: {df.columns.tolist()}")
    print(df.head(2))
    print()

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate some basic analysis: 
- Filtering records based on survey responses
- Normalizing a numeric score
- Grouping by a demographic attribute

Be sure to reference columns by their Croissant field/column `@id`. Below we select "GAD-7 Total Score" (commonly present in mental health surveys) and group by "gender" if available. 
Adjust field selection with your actual overview (see available fields and their `@id` in previous steps).


In [ ]:
# Choose which record set and fields to analyze
if len(dataframes) == 0:
    raise ValueError("No dataframes were loaded. Check previous steps.")

# Automatically pick the first available record set
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Print available columns to help identify field IDs
print(f"Columns in {main_record_set_id}:\n{df.columns.tolist()}")

###
# Manual step: Use your field overview to choose one numeric and one categorical field's @id
# For demonstration, we'll try common ones; replace as needed!
possible_numeric_fields = [col for col in df.columns if ('gad' in col.lower() or 'phq' in col.lower() or 'score' in col.lower() or 'total' in col.lower()) and pd.api.types.is_numeric_dtype(df[col])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # If not auto-found, just use the first available field
    numeric_field = df.columns[0]

# Try a categorical/group field
possible_group_fields = [col for col in df.columns if 'gend' in col.lower() or 'sex' in col.lower() or 'education' in col.lower() or 'age_group' in col.lower() or 'marital' in col.lower()]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]

# Preview distribution of numeric field
print(f"\nAnalyzing field: {numeric_field} (numeric)\nGrouping by: {group_field} (categorical)\n")

# Filter records where numeric_field > a threshold (pick 10 or the 25th percentile for demo)
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = 10 if df[numeric_field].max() > 10 else df[numeric_field].quantile(0.25)
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df[[group_field, numeric_field]].head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by demographic field and show means
    if group_field in filtered_df.columns and not pd.api.types.is_numeric_dtype(filtered_df[group_field]):
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped means of {numeric_field} by {group_field}:\n")
        display(grouped_df)
else:
    print(f"Field {numeric_field} is not numeric; can't demonstrate filtering and normalization.")

## 5. Visualization

Let's visualize the distribution of the selected score and the breakdown by a demographic attribute. 

We'll use matplotlib for basic visualization. Replace field names below by their actual `@id`s if needed.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot histogram of the main numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20, color="skyblue")
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot of score by group field
    if group_field in df.columns and not pd.api.types.is_numeric_dtype(df[group_field]):
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to fully explore a FAIR² survey dataset with Croissant metadata. We:
- Loaded schema and dataset
- Enumerated all record sets and fields with their Croissant `@id`
- Extracted records for tabular analysis
- Filtered, normalized, grouped, and visualized a mental health assessment score using only `@id` fields, as per best practice.

This demonstrates how standardized, FAIR-aligned metadata (Croissant) enables robust, reproducible, and transparent data science workflows.
